In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Data Cleaning and Preparation

This notebook prepares the raw solar, weather, and Kaggle datasets for forecasting and anomaly detection.

## Notebook sections
1. Forecast data from DKASC and NASA weather sources.
2. Merge Kaggle generation and weather data.
3. Create a labeled Kaggle anomaly dataset.

All cleaned outputs are saved in the `data/processed` directory.

## 1. Forecast data preparation

Load DKASC, convert timestamps, align observations by hour, and save `forecast_dkasc_clean.csv`.

In [14]:
dkasc_path = r'F:\Abdelrahman1\ME\programming\SIC\Project3.1\data\raw\Alice_Springs_2025.csv'

df = pd.read_csv(dkasc_path, nrows=2)

print("Irradiance / GHI:")
for col in df.columns:
    if 'Irradiance' in col or 'GHI' in col or 'Radiation' in col:
        print(f"- {col}")

print("\nTemperature:")
for col in df.columns:
    if 'Temperature' in col or 'Temp' in col:
        print(f"- {col}")

Irradiance / GHI:
- 101_DKA_WeatherStation_Global_Horizontal_Radiation
- 101_DKA_WeatherStation_Diffuse_Horizontal_Radiation
- 101_DKA_WeatherStation_Radiation_Global_Tilted
- 101_DKA_WeatherStation_Radiation_Diffuse_Tilted

Temperature:
- 101_DKA_WeatherStation_Weather_Temperature_Celsius


In [15]:
RAW_DIR = r'F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\raw'
PROCESSED_DIR = r'F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed\Forecast'
dkasc_path = os.path.join(RAW_DIR, 'Alice_Springs_2025.csv')

df = pd.read_csv(dkasc_path, on_bad_lines='skip', low_memory=False)

pv_col = '241_DKA_Totals_PV_Active_Power'

df_power = df[['timestamp', pv_col]].copy()
df_power.rename(columns={pv_col: 'actual_power_kw'}, inplace=True)

df_power['timestamp'] = pd.to_datetime(df_power['timestamp'], errors='coerce')
df_power['actual_power_kw'] = pd.to_numeric(df_power['actual_power_kw'], errors='coerce')
df_power['actual_power_kw'] = df_power['actual_power_kw'].clip(lower=0) 
df_power = df_power.dropna().sort_values('timestamp')

weather_cols = [
    '101_DKA_WeatherStation_Global_Horizontal_Radiation',
    '101_DKA_WeatherStation_Radiation_Global_Tilted',
    '101_DKA_WeatherStation_Weather_Temperature_Celsius'
]
df_weather = df[['timestamp'] + weather_cols].copy()
df_weather.rename(columns={
    '101_DKA_WeatherStation_Global_Horizontal_Radiation': 'ghi',
    '101_DKA_WeatherStation_Radiation_Global_Tilted': 'tilted_radiation',
    '101_DKA_WeatherStation_Weather_Temperature_Celsius': 'temperature'
}, inplace=True)

df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'], errors='coerce')

for col in ['ghi', 'tilted_radiation', 'temperature']:
    df_weather[col] = pd.to_numeric(df_weather[col], errors='coerce')
df_weather = df_weather.dropna().sort_values('timestamp')

df_merged = pd.merge_asof(
    df_power, df_weather, on='timestamp', direction='nearest', tolerance=pd.Timedelta('15min')
)
df_merged = df_merged.dropna()

clean_path = os.path.join(PROCESSED_DIR, 'forecast_dkasc_clean.csv')
df_merged.to_csv(clean_path, index=False)

C:\Users\HP\AppData\Local\Temp\ipykernel_16700\1226237175.py:36: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  df_power, df_weather, on='timestamp', direction='nearest', tolerance=pd.Timedelta('15min')


## 2. Kaggle generation and weather merge

Merge Plant 1 and Plant 2 generation data with their weather sensor data, then save `kaggle_merged_clean.csv`.

In [16]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "kaggle"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def process_kaggle_plant(plant_num):
    print(f"Preparing plant {plant_num}...")

    gen_path = RAW_DIR / f"Plant_{plant_num}_Generation_Data.csv"
    weather_path = RAW_DIR / f"Plant_{plant_num}_Weather_Sensor_Data.csv"

    df_gen = pd.read_csv(gen_path)
    df_weather = pd.read_csv(weather_path)

    df_gen["DATE_TIME"] = pd.to_datetime(df_gen["DATE_TIME"], format="mixed", dayfirst=True)
    df_weather["DATE_TIME"] = pd.to_datetime(df_weather["DATE_TIME"], format="mixed", dayfirst=True)

    df_merged = pd.merge(
        df_gen,
        df_weather,
        on=["DATE_TIME", "PLANT_ID"],
        how="inner",
        suffixes=("", "_weather"),
    )
    return df_merged


if __name__ == "__main__":
    plant1 = process_kaggle_plant(1)
    plant2 = process_kaggle_plant(2)

    df_kaggle_all = pd.concat([plant1, plant2], ignore_index=True)

    print("Both plants were merged successfully!")
    print(f"Number of rows: {len(df_kaggle_all):,}")
    print(f"Number of columns: {len(df_kaggle_all.columns):,}")

    cols_to_show = [
        "DATE_TIME", "PLANT_ID", "SOURCE_KEY", "AC_POWER",
        "AMBIENT_TEMPERATURE", "IRRADIATION",
    ]
    print("\nData sample:")
    print(df_kaggle_all[cols_to_show].head())

    output_path = PROCESSED_DIR / "Anomaly\\kaggle_merged_clean.csv"
    df_kaggle_all.to_csv(output_path, index=False)
    print(f"\nSaved kaggle_merged_clean.csv to: {output_path}")

Preparing plant 1...
Preparing plant 2...
Both plants were merged successfully!
Number of rows: 136,472
Number of columns: 11

Data sample:
   DATE_TIME  PLANT_ID       SOURCE_KEY  AC_POWER  AMBIENT_TEMPERATURE  \
0 2020-05-15   4135001  1BY6WEcLGh8j5v7       0.0            25.184316   
1 2020-05-15   4135001  1IF53ai7Xc0U56Y       0.0            25.184316   
2 2020-05-15   4135001  3PZuoBAID5Wc2HD       0.0            25.184316   
3 2020-05-15   4135001  7JYdWkrLSPkdwr4       0.0            25.184316   
4 2020-05-15   4135001  McdE0feGgRqW7Ca       0.0            25.184316   

   IRRADIATION  
0          0.0  
1          0.0  
2          0.0  
3          0.0  
4          0.0  

Saved kaggle_merged_clean.csv to: f:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed\Anomaly\kaggle_merged_clean.csv


## 3. Synthetic anomaly dataset

Filter daylight observations, create anomaly labels, inject representative inverter faults, and save `kaggle_anomaly_data.csv`.

In [17]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
input_path = PROCESSED_DIR / "Anomaly\\kaggle_merged_clean.csv"
output_path = PROCESSED_DIR / "Anomaly\\kaggle_anomaly_data.csv"

df = pd.read_csv(input_path)
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])

print("Filtering daylight observations...")
df_daylight = df[df["IRRADIATION"] > 0.05].copy()
df_daylight["is_anomaly"] = 0

df_daylight = df_daylight.set_index(["DATE_TIME", "SOURCE_KEY"]).sort_index()

inverters = df_daylight.index.get_level_values("SOURCE_KEY").unique()
np.random.seed(42)
selected_inverters = np.random.choice(inverters, size=15, replace=False)

print("Injecting faults into 15 inverters...")
days = df_daylight.index.get_level_values("DATE_TIME").normalize().unique()
selected_days = np.random.choice(days, size=5, replace=False)

for inv in selected_inverters:
    day = np.random.choice(selected_days)

    mask = (
        (df_daylight.index.get_level_values("SOURCE_KEY") == inv)
        & (df_daylight.index.get_level_values("DATE_TIME").normalize() == day)
    )
    # (Inverter Dropout) 33% of the faults result in the inverter being disconnected for 4 hours.
    if np.random.rand() < 0.33:
        times = df_daylight[mask].index.get_level_values("DATE_TIME")
        if len(times) > 16:
            bad_times = times[len(times) // 3:len(times) // 3 + 16]
            df_daylight.loc[(bad_times, inv), "AC_POWER"] = 0
            df_daylight.loc[(bad_times, inv), "DC_POWER"] = 0
            df_daylight.loc[(bad_times, inv), "is_anomaly"] = 1
    # (String Failure) 33% of breakdowns reduce production by half (50%).
    elif np.random.rand() < 0.66:
        times = df_daylight[mask].index.get_level_values("DATE_TIME")
        if len(times) > 24:
            bad_times = times[len(times) // 4:len(times) // 4 + 24]
            df_daylight.loc[(bad_times, inv), "AC_POWER"] *= 0.5
            df_daylight.loc[(bad_times, inv), "is_anomaly"] = 1
    # (Soiling) 34% of malfunctions are due to dust accumulation (a 15% reduction).
    else:
        df_daylight.loc[mask, "AC_POWER"] *= 0.85
        times = df_daylight[mask].index.get_level_values("DATE_TIME")
        if len(times) > 10:
            mid_times = times[len(times) // 2:len(times) // 2 + 10]
            df_daylight.loc[(mid_times, inv), "is_anomaly"] = 1

df_final = df_daylight.reset_index()
df_final = df_final[df_final["AC_POWER"] >= 0]

print("Faults injected successfully!")
print(f"Total rows: {len(df_final):,}")
print(f"Normal rows: {len(df_final[df_final['is_anomaly'] == 0]):,}")
print(f"Anomaly rows: {len(df_final[df_final['is_anomaly'] == 1]):,}")

df_final.to_csv(output_path, index=False)
print(f"Saved kaggle_anomaly_data.csv to: {output_path}")

Filtering daylight observations...
Injecting faults into 15 inverters...
Faults injected successfully!
Total rows: 64,892
Normal rows: 64,608
Anomaly rows: 284
Saved kaggle_anomaly_data.csv to: f:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed\Anomaly\kaggle_anomaly_data.csv


----------------------------------------

----------------------------------------

In [18]:
import pandas as pd
kaggle_anomaly = pd.read_csv("F:\\Abdelrahman1\\ME\\programming\\SIC\\Project3.2\\data\\processed\\Anomaly\\kaggle_anomaly_data.csv")
kaggle_anomaly.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64892 entries, 0 to 64891
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DATE_TIME            64892 non-null  object 
 1   SOURCE_KEY           64892 non-null  object 
 2   PLANT_ID             64892 non-null  int64  
 3   DC_POWER             64892 non-null  float64
 4   AC_POWER             64892 non-null  float64
 5   DAILY_YIELD          64892 non-null  float64
 6   TOTAL_YIELD          64892 non-null  float64
 7   SOURCE_KEY_weather   64892 non-null  object 
 8   AMBIENT_TEMPERATURE  64892 non-null  float64
 9   MODULE_TEMPERATURE   64892 non-null  float64
 10  IRRADIATION          64892 non-null  float64
 11  is_anomaly           64892 non-null  int64  
dtypes: float64(7), int64(2), object(3)
memory usage: 5.9+ MB


In [19]:
kaggle_anomaly.head()

,DATE_TIME,SOURCE_KEY,PLANT_ID,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD,SOURCE_KEY_weather,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,is_anomaly
0,2020-05-15 06:45:00,1BY6WEcLGh8j5v7,4135001,1224.750000,119.275000,34.875000,6.259594e+06,HmiyD2TTLFNqkNe,24.218990,24.442444,0.095394,0
1,2020-05-15 06:45:00,1IF53ai7Xc0U56Y,4135001,1334.285714,130.142857,39.142857,6.183684e+06,HmiyD2TTLFNqkNe,24.218990,24.442444,0.095394,0
2,2020-05-15 06:45:00,3PZuoBAID5Wc2HD,4135001,1306.714286,127.414286,37.000000,6.987796e+06,HmiyD2TTLFNqkNe,24.218990,24.442444,0.095394,0
3,2020-05-15 06:45:00,4UPUqMRk7TRMgml,4136001,158.013333,154.486667,39.466667,2.429050e+06,iq8k7ZNt4Mwm3w0,25.048245,25.746662,0.103496,0
4,2020-05-15 06:45:00,7JYdWkrLSPkdwr4,4135001,1338.125000,130.487500,39.000000,7.602999e+06,HmiyD2TTLFNqkNe,24.218990,24.442444,0.095394,0


In [20]:
kaggle_anomaly.isnull().sum()

DATE_TIME              0
SOURCE_KEY             0
PLANT_ID               0
DC_POWER               0
AC_POWER               0
DAILY_YIELD            0
TOTAL_YIELD            0
SOURCE_KEY_weather     0
AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
is_anomaly             0
dtype: int64

In [21]:
print(kaggle_anomaly.duplicated().sum())

0


In [22]:
def FindAndHandelQutlierInData(data, col):
    clean_col = data[col].dropna()
    Q1 = np.quantile(clean_col, .25)
    Q3 = np.quantile(clean_col, .75)
    IQR = Q3 - Q1
    lower = max(0, Q1 - 1.5 * IQR)
    upper = Q3 + 1.5 * IQR

    NoOutlier = ((clean_col > upper) | (clean_col < lower)).sum()
    print(f"{col:<19}: found {NoOutlier:<9} outlier(s) -> bounds = [{lower:.2f}, {upper:.2f}]")

for col in kaggle_anomaly.columns:
    if kaggle_anomaly[col].dtype in ['Int64', 'float64']:
        FindAndHandelQutlierInData(kaggle_anomaly, col)

DC_POWER           : found 0         outlier(s) -> bounds = [0.00, 15887.25]
AC_POWER           : found 0         outlier(s) -> bounds = [0.00, 1773.01]
DAILY_YIELD        : found 0         outlier(s) -> bounds = [0.00, 11889.30]
TOTAL_YIELD        : found 12935     outlier(s) -> bounds = [0.00, 513478605.39]
AMBIENT_TEMPERATURE: found 0         outlier(s) -> bounds = [18.45, 39.98]
MODULE_TEMPERATURE : found 0         outlier(s) -> bounds = [8.61, 74.52]
IRRADIATION        : found 0         outlier(s) -> bounds = [0.00, 1.41]


In [23]:
import pandas as pd
forecast = pd.read_csv("F:\\Abdelrahman1\\ME\\programming\\SIC\\Project3.2\\data\\processed\\Forecast\\forecast_dkasc_clean.csv")
forecast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98146 entries, 0 to 98145
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   timestamp         98146 non-null  object 
 1   actual_power_kw   98146 non-null  float64
 2   ghi               98146 non-null  float64
 3   tilted_radiation  98146 non-null  float64
 4   temperature       98146 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.7+ MB


In [24]:
forecast.head()

,timestamp,actual_power_kw,ghi,tilted_radiation,temperature
0,2025-01-01 00:00:00,0.0,2.122361,4.169994,25.154861
1,2025-01-01 00:05:00,0.0,2.309545,4.253466,25.035696
2,2025-01-01 00:10:00,0.0,2.226635,4.155552,24.824100
3,2025-01-01 00:15:00,0.0,2.177083,4.063492,24.623623
4,2025-01-01 00:20:00,0.0,2.419703,4.036308,24.387362


In [25]:
forecast.isnull().sum()

timestamp           0
actual_power_kw     0
ghi                 0
tilted_radiation    0
temperature         0
dtype: int64

In [26]:
forecast.duplicated().sum()

np.int64(0)

In [27]:
def FindAndHandelQutlierInData(data, col):
    clean_col = data[col].dropna()
    Q1 = np.quantile(clean_col, .25)
    Q3 = np.quantile(clean_col, .75)
    IQR = Q3 - Q1
    lower = max(0, Q1 - 1.5 * IQR)
    upper = Q3 + 1.5 * IQR

    NoOutlier = ((clean_col > upper) | (clean_col < lower)).sum()
    print(f"{col:<19}: found {NoOutlier:<9} outlier(s) -> bounds = [{lower:.2f}, {upper:.2f}]")

for col in kaggle_anomaly.columns:
    if kaggle_anomaly[col].dtype in ['Int64', 'float64']:
        FindAndHandelQutlierInData(kaggle_anomaly, col)

DC_POWER           : found 0         outlier(s) -> bounds = [0.00, 15887.25]
AC_POWER           : found 0         outlier(s) -> bounds = [0.00, 1773.01]
DAILY_YIELD        : found 0         outlier(s) -> bounds = [0.00, 11889.30]
TOTAL_YIELD        : found 12935     outlier(s) -> bounds = [0.00, 513478605.39]
AMBIENT_TEMPERATURE: found 0         outlier(s) -> bounds = [18.45, 39.98]
MODULE_TEMPERATURE : found 0         outlier(s) -> bounds = [8.61, 74.52]
IRRADIATION        : found 0         outlier(s) -> bounds = [0.00, 1.41]


## Outlier Review and Final Cleaning Decision

The IQR method identified outliers in several numerical columns, but statistical outliers were not automatically removed. In a solar-energy project, some values are physically valid:

- A temperature near `50°C` can be a normal environmental condition in Alice Springs.
- `TOTAL_YIELD` naturally increases over time, so a large value is not necessarily a fault.
- Zero or low power output is valid during nighttime or under low irradiance.

As a result, the dataset was cleaned by removing unnecessary columns and retaining physically meaningful values. The anomaly model will use injected fault labels and engineered features to distinguish normal conditions from inverter failure. This approach avoids deleting real events and keeps the cleaned data suitable for both forecasting and anomaly detection.